## 0 · Sync with GitHub
I run this at the start of every session so I'm always working off the latest code.

In [ ]:
# ── Pull latest code from GitHub (run at the start of every session) ──────────
import os

if os.path.exists("DSC_599_Semester_Project"):
    # Repo already cloned — just pull the latest changes
    %cd DSC_599_Semester_Project
    !git pull origin colab
else:
    # First time — clone and switch to the colab branch
    !git clone https://github.com/ConlerSimmons/DSC_599_Semester_Project.git
    %cd DSC_599_Semester_Project
    !git checkout colab

print("\nCurrent branch:")
!git branch --show-current
print("Latest commit:")
!git log --oneline -1

# Graph Neural Network — Relational Fraud Detection

The idea behind this model is that fraud isn't random — fraudsters reuse the same cards, devices, and email accounts across multiple transactions. If I can connect those transactions in a graph, I can let the model propagate fraud signals between related nodes rather than treating every transaction in isolation.

Every transaction is a node. I connect two nodes with an edge if they share a value in any of these identity columns: `card1`, `card4`, `card6`, `addr1`, `P_emaildomain`, `R_emaildomain`, `id_30`, `id_31`, or `DeviceInfo`. The intuition is that if transaction A and B used the same card, they're probably from the same person — and if one is fraud, that's useful information about the other.

I tried k-NN edges initially (connecting transactions that are numerically similar) but abandoned it — computing nearest neighbours on 590k × 50 features is O(N²) and would take hours just to build the graph. Identity edges are O(N) and more directly encode the fraud hypothesis anyway.

The model itself is a 3-layer residual GNN with mean aggregation. Each layer aggregates neighbour features, applies a linear transformation, and adds a residual connection so gradients can flow cleanly during backprop.

In [ ]:
# Install required packages (run once in Colab)
!pip install -q torch pandas numpy scikit-learn pyarrow lightgbm

## 1 · Data Setup
Same setup as the TabTransformer notebook — pulling from Drive and copying to local storage for faster reads.

In [ ]:
import os
from google.colab import drive

# ── Mount Google Drive and point to your data ─────────────────────────────────
# Before running:
#   1. Upload train_transaction.csv and train_identity.csv to Google Drive
#      e.g. into a folder called "ieee_fraud/raw/"
#   2. Update DRIVE_DATA_PATH below to match where you put them
#   3. Run this cell — it will prompt you to authorise Drive access

drive.mount("/content/drive")

DRIVE_DATA_PATH = "/content/drive/MyDrive/Data_Fraud"  # ← update if your folder name differs
DATA_DIR = "data"

os.makedirs(f"{DATA_DIR}/raw",     exist_ok=True)
os.makedirs(f"{DATA_DIR}/interim", exist_ok=True)

# Copy from Drive to Colab local storage (faster reads during training)
os.system(f"cp {DRIVE_DATA_PATH}/train_transaction.csv {DATA_DIR}/raw/")
os.system(f"cp {DRIVE_DATA_PATH}/train_identity.csv    {DATA_DIR}/raw/")

print("Files ready:")
os.system(f"ls -lh {DATA_DIR}/raw/")

## 2 · Imports
I force the device selection here explicitly. The GNN runs full-graph training — the entire 590k-node graph lives in memory at once — so I need to be deliberate about where tensors land.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

# GNN uses full-graph training — the entire 590k-node graph is passed in one
# forward call. On Colab T4/A100, CUDA should have enough VRAM (~4-8 GB needed).
# Falls back to CPU if CUDA is unavailable (e.g. running locally on Apple Silicon).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3 · Load & Merge Data
Same loading logic as the TabTransformer — joins transactions with identity records and caches the result.

In [ ]:
import os
import pandas as pd

def load_merged_train(data_dir="data"):
    """
    Load and merge the IEEE-CIS training files.

    Reads train_transaction.csv and train_identity.csv from data_dir/raw/,
    left-joins them on TransactionID so every transaction is kept (identity
    features are NaN for ~76% of rows that have no identity record), then
    caches the result to data_dir/interim/merged_train.parquet for fast
    re-loads.

    Returns
    -------
    pd.DataFrame  shape ~ (590540, 434)
    """
    cache = os.path.join(data_dir, "interim", "merged_train.parquet")
    if os.path.exists(cache):
        print(f"Loading from cache: {cache}")
        df = pd.read_parquet(cache)
        print(f"Shape: {df.shape}")
        return df

    raw = os.path.join(data_dir, "raw")
    print("Reading train_transaction.csv …")
    trans = pd.read_csv(os.path.join(raw, "train_transaction.csv"))
    print("Reading train_identity.csv …")
    ident = pd.read_csv(os.path.join(raw, "train_identity.csv"))
    print("Merging on TransactionID …")
    df = trans.merge(ident, how="left", on="TransactionID")
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    df.to_parquet(cache, index=False)
    print(f"Cached to {cache}.  Shape: {df.shape}")
    return df

df = load_merged_train(DATA_DIR)
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(f"Sorted by TransactionDT.  Shape: {df.shape}")

## 4 · Temporal Split
I define the split before running feature selection so that when LightGBM fits to rank features, it only sees training data. Same 70/15/15 split as the TabTransformer so the comparison is apples-to-apples.

In [ ]:
# Temporal 70 / 15 / 15 split — respects transaction time ordering.
# Using random splits on time-series data leaks future info into training.
n        = len(df)
n_train  = int(0.70 * n)
n_val    = int(0.15 * n)
train_idx = list(range(0, n_train))
val_idx   = list(range(n_train, n_train + n_val))
test_idx  = list(range(n_train + n_val, n))
print(f"Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}")

## 5 · Feature Selection
I use the exact same LightGBM importance-based selection as the TabTransformer. Both models get the same 50 numeric + 20 categorical features — if I gave them different features the comparison would be meaningless.

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

def select_features_by_importance(df, train_idx, target_col="isFraud",
                                   max_numeric=50, max_categorical=20):
    """
    Rank all candidate features by LightGBM split importance, then pick the top N.

    Why this beats heuristic selection:
      The IEEE-CIS dataset has 339 anonymised V-columns — their names reveal
      nothing about their value. Heuristic selection (first 50 columns) grabs
      V1-V11, which may be less informative than V45 or V258. LightGBM quickly
      identifies which features the trees actually use for splits.

    Process:
      1. Drop columns that are >95% missing.
      2. Classify remaining columns as numeric or categorical.
      3. Fit a fast LightGBM (100 trees) on the TRAINING split only.
      4. Rank features by cumulative split gain.
      5. Return top max_numeric numeric + top max_categorical categorical.
    """
    ignore = {target_col, "TransactionID"}
    na_frac = df.isna().mean()
    kept = [c for c in na_frac[na_frac < 0.95].index if c not in ignore]

    num_cands, cat_cands = [], []
    for col in kept:
        dt = df[col].dtype
        if dt == "object" or str(dt) == "category":
            cat_cands.append(col)
        elif np.issubdtype(dt, np.integer):
            (cat_cands if df[col].nunique(dropna=True) <= 20 else num_cands).append(col)
        elif np.issubdtype(dt, np.floating):
            num_cands.append(col)

    all_cands = num_cands + cat_cands
    X = df[all_cands].copy()
    for col in cat_cands:
        X[col] = X[col].astype("category").cat.codes  # -1 for NaN, fine for trees
    X = X.fillna(-999).values.astype(np.float32)
    y = df[target_col].values

    pos = (y[train_idx] == 1).sum()
    neg = (y[train_idx] == 0).sum()
    print("Running LightGBM for feature importance (100 trees) …")
    lgb = LGBMClassifier(n_estimators=100, n_jobs=-1, verbose=-1,
                          random_state=42, scale_pos_weight=neg/pos)
    lgb.fit(X[train_idx], y[train_idx])

    importance = pd.Series(lgb.feature_importances_, index=all_cands).sort_values(ascending=False)
    num_set, cat_set = set(num_cands), set(cat_cands)
    numeric_cols     = [c for c in importance.index if c in num_set][:max_numeric]
    categorical_cols = [c for c in importance.index if c in cat_set][:max_categorical]

    print(f"Selected {len(numeric_cols)} numeric + {len(categorical_cols)} categorical by importance")
    print("Top 10 numeric:   ", numeric_cols[:10])
    print("Categorical:      ", categorical_cols)
    return numeric_cols, categorical_cols

numeric_cols, categorical_cols = select_features_by_importance(
    df, train_idx, target_col="isFraud", max_numeric=50, max_categorical=20
)

## 6 · Build the Transaction Graph
This is the step that makes the GNN fundamentally different from the TabTransformer. I'm building an `edge_index` tensor — a (2, E) matrix where each column is one directed edge — that encodes which transactions are related to which.

For each identity column I group transactions by shared value and connect them. Small groups (10 or fewer) get fully connected into a clique. Larger groups get a hub-and-spoke structure with a chain connecting sequential members — this keeps the edge count linear in group size rather than quadratic, which matters a lot at 590k nodes.

In [ ]:
def build_transaction_graph(df, min_group_size=2, max_group_size=1000,
                             small_group_full_connect=10):
    """
    Builds the transaction graph from shared identity columns.

    I skip NaN groups deliberately — if two transactions both lack a card number,
    that doesn't mean they're related, it just means we don't have the data.
    Connecting unknown-to-unknown would add noise rather than signal.

    Self-loops are added so each node aggregates its own features alongside
    its neighbours during message passing — without them a node has no way
    to preserve its own representation across layers.
    """
    df = df.reset_index(drop=True)
    src_list, dst_list = [], []

    id_cols = ["card1","card4","card6","addr1","P_emaildomain",
               "R_emaildomain","id_30","id_31","DeviceInfo"]
    active_cols = [c for c in id_cols if c in df.columns]

    for col in active_cols:
        vals   = df[col].astype(str)
        groups = vals.groupby(vals).groups
        for val, idxs in groups.items():
            if val in ("nan","None",""): continue
            grp  = sorted(list(idxs))
            size = len(grp)
            if size < min_group_size or size > max_group_size: continue

            if size <= small_group_full_connect:
                # Full clique — every pair connected
                for i in range(size):
                    for j in range(i + 1, size):
                        src_list += [grp[i], grp[j]]
                        dst_list += [grp[j], grp[i]]
            else:
                # Hub + chain pattern — keeps edge count linear in group size
                hub = grp[0]
                for other in grp[1:]:
                    src_list += [hub, other]; dst_list += [other, hub]
                for i in range(size - 1):
                    src_list += [grp[i], grp[i+1]]; dst_list += [grp[i+1], grp[i]]

    if not src_list:
        return torch.empty((2, 0), dtype=torch.long)

    ei = torch.tensor([src_list, dst_list], dtype=torch.long)
    ei = torch.unique(ei.t(), dim=0).t().contiguous()   # deduplicate

    # Self-loops: each node aggregates its own features
    n   = len(df)
    sl  = torch.arange(n, dtype=torch.long)
    ei  = torch.cat([ei, torch.stack([sl, sl])], dim=1)
    return ei

print("Building transaction graph …")
edge_index = build_transaction_graph(df)
print(f"edge_index shape: {edge_index.shape}  ({edge_index.shape[1]:,} edges)")

## 7 · Model Architecture
The model has two distinct phases. First, I encode each transaction's raw features into a shared embedding space using a numeric projection and per-column categorical embeddings. Then I run three rounds of message passing where each node aggregates the mean of its neighbours' representations, applies a linear transformation, and adds a residual connection back to itself.

The residual connections are important here — without them, deep GNNs tend to "over-smooth" where every node ends up with roughly the same representation regardless of its local neighbourhood. The LayerNorm after each layer helps with training stability.

In [ ]:
class SimpleGNN(nn.Module):
    """
    My 3-layer residual GNN for node-level fraud classification.

    I set hidden_dim=128 rather than 256 — at 590k nodes the graph structure
    is doing the heavy lifting, not raw model capacity, and the smaller size
    keeps memory manageable for full-graph training on a single GPU.
    """

    def __init__(self, num_numeric, num_categories_per_col,
                 embed_dim=32, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(size + 1, embed_dim) for size in num_categories_per_col
        ])
        self.num_linear   = nn.Linear(num_numeric, embed_dim)
        total_in          = embed_dim * (1 + len(num_categories_per_col))
        self.input_linear = nn.Linear(total_in, hidden_dim)

        # Three residual GNN layers
        self.gcn1, self.gcn2, self.gcn3 = (
            nn.Linear(hidden_dim, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout    = nn.Dropout(dropout)
        self.act        = nn.ReLU()
        self.out_linear = nn.Linear(hidden_dim, 1)

    def _mean_agg(self, x, edge_index):
        """
        Mean-aggregate neighbour features for each node.
        Fast O(E) scatter operation — no intermediate edge tensors.
        """
        if edge_index.numel() == 0:
            return x
        src, dst = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        deg = torch.zeros(x.size(0), device=x.device).index_add_(
            0, dst, torch.ones(len(dst), device=x.device)
        ).clamp_min(1.0).unsqueeze(1)
        return agg / deg

    def forward(self, x_num, x_cat, edge_index):
        # Encode features into a shared embedding space
        h = torch.cat(
            [self.act(self.num_linear(x_num))] +
            [emb(x_cat[:, i].clamp(0, emb.num_embeddings - 1))
             for i, emb in enumerate(self.cat_embeddings)],
            dim=1,
        )
        h = self.act(self.input_linear(h))

        # Three residual message-passing layers
        for gcn, norm in [(self.gcn1, self.norm1),
                          (self.gcn2, self.norm2),
                          (self.gcn3, self.norm3)]:
            h_res = h
            h     = self.dropout(self.act(gcn(self._mean_agg(h, edge_index))))
            h     = norm(h + h_res)

        return self.out_linear(h).squeeze(-1)

## 8 · Training Loop
The biggest difference from the TabTransformer is that I can't mini-batch here. The GNN needs the full graph in memory at once because message passing requires knowing every node's neighbours — if I only loaded a subset of nodes, the neighbourhood information would be incomplete. That means every forward pass touches all 590k nodes and all edges.

Because of this, I use a higher early stopping patience (20 epochs) — GNNs converge more slowly than mini-batch models and I don't want to stop prematurely. Same label smoothing and F2 threshold tuning as the TabTransformer.

In [ ]:
def train_gnn(df, numeric_cols, categorical_cols, target_col="isFraud",
              num_epochs=100, lr=1e-3, train_idx=None, val_idx=None,
              test_idx=None, early_stopping_patience=20):
    """
    Full-graph training of the SimpleGNN.

    I save the best model state to CPU after each improvement so I'm not
    accumulating multiple copies of the weights in GPU memory across epochs.
    At restore time I move the state back to the device before loading.
    """
    cols = numeric_cols + categorical_cols + [target_col]
    df   = df[cols].copy().reset_index(drop=True)

    # ── Numeric scaling ───────────────────────────────────────────────────────
    num_df = df[numeric_cols].fillna(0.0).astype("float32")
    num_df = (num_df - num_df.mean()) / num_df.std().replace(0, 1.0)
    df[numeric_cols] = num_df
    x_num = torch.tensor(num_df.values, dtype=torch.float32)

    # ── Categorical encoding ──────────────────────────────────────────────────
    cat_sizes, cat_arrays = [], []
    for col in categorical_cols:
        vals    = df[col].astype(str)
        mapping = {v: i for i, v in enumerate(sorted(vals.unique()))}
        cat_sizes.append(len(mapping))
        cat_arrays.append(vals.map(mapping).astype("int64").values)

    x_cat = (torch.tensor(np.stack(cat_arrays, axis=1), dtype=torch.long)
             if cat_arrays else torch.empty((len(df), 0), dtype=torch.long))

    y = torch.tensor(df[target_col].values.astype("float32"), dtype=torch.float32)

    # ── Graph ─────────────────────────────────────────────────────────────────
    if "edge_index" not in dir():
        print("Building graph …")
    ei = build_transaction_graph(df)

    # ── Splits ────────────────────────────────────────────────────────────────
    if train_idx is None or val_idx is None:
        n_tr = int(0.8 * len(df))
        train_idx = list(range(0, n_tr)); val_idx = list(range(n_tr, len(df)))
    tr  = torch.tensor(train_idx, dtype=torch.long)
    val = torch.tensor(val_idx,   dtype=torch.long)
    tst = torch.tensor(test_idx,  dtype=torch.long) if test_idx else None

    # Move to device
    x_num, x_cat, y, ei = x_num.to(device), x_cat.to(device), y.to(device), ei.to(device)

    # ── Model ─────────────────────────────────────────────────────────────────
    # hidden_dim=128 (down from 256) — halves memory, ~2x faster per epoch,
    # modest accuracy tradeoff. At 590k nodes the graph structure does the
    # heavy lifting, not model capacity.
    model = SimpleGNN(len(numeric_cols), cat_sizes, hidden_dim=128).to(device)
    pos   = (y[tr] == 1).sum(); neg = (y[tr] == 0).sum()
    criterion = nn.BCEWithLogitsLoss(pos_weight=(neg / pos).clamp(min=1.0))
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # ── Training loop ─────────────────────────────────────────────────────────
    best_pr, best_epoch, pat_ctr, best_state = -1.0, 0, 0, None

    for epoch in range(1, num_epochs + 1):
        model.train(); optimizer.zero_grad()
        logits   = model(x_num, x_cat, ei)
        # Label smoothing: softens 0→0.05 and 1→0.95 to reduce overconfidence
        smooth   = 0.05
        y_smooth = y[tr] * (1 - smooth) + smooth * 0.5
        loss     = criterion(logits[tr], y_smooth)
        if not torch.isfinite(loss): print(f"Non-finite loss at epoch {epoch}"); break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(x_num, x_cat, ei))[val].cpu().numpy()
        ep_pr = average_precision_score(y[val].cpu().numpy(), vp)
        print(f"[Epoch {epoch:3d}] loss={loss.item():.4f}  val_pr_auc={ep_pr:.4f}")

        if ep_pr > best_pr:
            best_pr, best_epoch, pat_ctr = ep_pr, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            pat_ctr += 1
            if pat_ctr >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch} (pr_auc={best_pr:.4f})")
                break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f"Restored best model from epoch {best_epoch}")

    # ── Evaluation ────────────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(x_num, x_cat, ei))

    def eval_split(idx_tensor, label):
        yt = y[idx_tensor].cpu().numpy()
        ys = probs[idx_tensor].cpu().numpy()
        precs, recs, thrs = precision_recall_curve(yt, ys)
        # F2 score — weights recall 2× over precision for fraud detection
        beta = 2
        f2s  = (1 + beta**2) * precs[:-1] * recs[:-1] / (
            beta**2 * precs[:-1] + recs[:-1] + 1e-8
        )
        thr = float(thrs[f2s.argmax()])
        yp  = (ys >= thr).astype("int32")
        return {
            f"{label}_precision": precision_score(yt, yp, zero_division=0),
            f"{label}_recall":    recall_score(yt, yp, zero_division=0),
            f"{label}_f1":        f1_score(yt, yp, zero_division=0),
            f"{label}_roc_auc":   roc_auc_score(yt, ys),
            f"{label}_pr_auc":    average_precision_score(yt, ys),
            f"{label}_threshold": thr,
        }

    metrics = eval_split(val, "val")
    print("\n===== GNN — Validation =====")
    for k, v in metrics.items(): print(f"  {k:22s}: {v:.4f}")

    if tst is not None:
        test_m = eval_split(tst, "test")
        print("\n===== GNN — Test =====")
        for k, v in test_m.items(): print(f"  {k:22s}: {v:.4f}")
        metrics.update(test_m)

    return metrics, model

## 9 · Train the Model
Graph construction takes a few minutes — it's scanning all 590k rows for shared identity values. Training itself should be around 5 seconds per epoch on CUDA. Early stopping will likely kick in well before epoch 100.

In [ ]:
# Graph construction: ~2-3 min. Training: ~5 sec/epoch. Expected total: ~10-15 min.
print("Building graph …")
edge_index = build_transaction_graph(df)
print(f"Graph built: {edge_index.shape[1]:,} edges")

gnn_metrics, gnn_model = train_gnn(
    df, numeric_cols, categorical_cols,
    target_col="isFraud",
    num_epochs=250,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
)

## 10 · Results
Val metrics guided training. Test metrics are what matter — the model never touched the test set until this final evaluation.

In [ ]:
print("\n" + "="*45)
print("  GNN — Final Results")
print("="*45)
for k, v in gnn_metrics.items():
    print(f"  {k:24s}: {v:.4f}")